In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import re

In [2]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("saurabhshahane/fake-news-classification")

print("Path to dataset files:", path)

100%|██████████| 92.1M/92.1M [00:00<00:00, 184MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/saurabhshahane/fake-news-classification/versions/77


In [3]:
import os

print(os.listdir(path))

['WELFake_Dataset.csv']


In [4]:
data = pd.read_csv(path + '/WELFake_Dataset.csv')

In [5]:
data.head()

,Unnamed: 0,title,text,label
0,0,LAW ENFORCEMENT ON HIGH ALERT Following Threat...,No comment is expected from Barack Obama Membe...,1
1,1,NaN,Did they post their votes for Hillary already?,1
2,2,UNBELIEVABLE! OBAMA’S ATTORNEY GENERAL SAYS MO...,"Now, most of the demonstrators gathered last ...",1
3,3,"Bobby Jindal, raised Hindu, uses story of Chri...",A dozen politically active pastors came here f...,0
4,4,SATAN 2: Russia unvelis an image of its terrif...,"The RS-28 Sarmat missile, dubbed Satan 2, will...",1


In [6]:
data.drop(columns = ['Unnamed: 0'], inplace = True)

In [7]:
data.head()

,title,text,label
0,LAW ENFORCEMENT ON HIGH ALERT Following Threat...,No comment is expected from Barack Obama Membe...,1
1,NaN,Did they post their votes for Hillary already?,1
2,UNBELIEVABLE! OBAMA’S ATTORNEY GENERAL SAYS MO...,"Now, most of the demonstrators gathered last ...",1
3,"Bobby Jindal, raised Hindu, uses story of Chri...",A dozen politically active pastors came here f...,0
4,SATAN 2: Russia unvelis an image of its terrif...,"The RS-28 Sarmat missile, dubbed Satan 2, will...",1


fake = 0
real = 1

In [8]:
data.isnull().sum()

,0
title,558
text,39
label,0


In [9]:
data

,title,text,label
0,LAW ENFORCEMENT ON HIGH ALERT Following Threat...,No comment is expected from Barack Obama Membe...,1
1,NaN,Did they post their votes for Hillary already?,1
2,UNBELIEVABLE! OBAMA’S ATTORNEY GENERAL SAYS MO...,"Now, most of the demonstrators gathered last ...",1
3,"Bobby Jindal, raised Hindu, uses story of Chri...",A dozen politically active pastors came here f...,0
4,SATAN 2: Russia unvelis an image of its terrif...,"The RS-28 Sarmat missile, dubbed Satan 2, will...",1
...,...,...,...
72129,Russians steal research on Trump in hack of U....,WASHINGTON (Reuters) - Hackers believed to be ...,0
72130,WATCH: Giuliani Demands That Democrats Apolog...,"You know, because in fantasyland Republicans n...",1
72131,Migrants Refuse To Leave Train At Refugee Camp...,Migrants Refuse To Leave Train At Refugee Camp...,0
72132,Trump tussle gives unpopular Mexican leader mu...,MEXICO CITY (Reuters) - Donald Trump’s combati...,0


In [10]:
data.fillna('', inplace = True)

In [11]:
(data['title'] + ' ' +  data['text'])[0]

'LAW ENFORCEMENT ON HIGH ALERT Following Threats Against Cops And Whites On 9-11By #BlackLivesMatter And #FYF911 Terrorists [VIDEO] No comment is expected from Barack Obama Members of the #FYF911 or #FukYoFlag and #BlackLivesMatter movements called for the lynching and hanging of white people and cops. They encouraged others on a radio show Tuesday night to  turn the tide  and kill white people and cops to send a message about the killing of black people in America.One of the F***YoFlag organizers is called  Sunshine.  She has a radio blog show hosted from Texas called,  Sunshine s F***ing Opinion Radio Show. A snapshot of her #FYF911 @LOLatWhiteFear Twitter page at 9:53 p.m. shows that she was urging supporters to  Call now!! #fyf911 tonight we continue to dismantle the illusion of white Below is a SNAPSHOT Twitter Radio Call Invite   #FYF911The radio show aired at 10:00 p.m. eastern standard time.During the show, callers clearly call for  lynching  and  killing  of white people.A 2:3

In [12]:
data['news'] = data['title'] + ' ' +  data['text']

In [13]:
data.head()

,title,text,label,news
0,LAW ENFORCEMENT ON HIGH ALERT Following Threat...,No comment is expected from Barack Obama Membe...,1,LAW ENFORCEMENT ON HIGH ALERT Following Threat...
1,,Did they post their votes for Hillary already?,1,Did they post their votes for Hillary already?
2,UNBELIEVABLE! OBAMA’S ATTORNEY GENERAL SAYS MO...,"Now, most of the demonstrators gathered last ...",1,UNBELIEVABLE! OBAMA’S ATTORNEY GENERAL SAYS MO...
3,"Bobby Jindal, raised Hindu, uses story of Chri...",A dozen politically active pastors came here f...,0,"Bobby Jindal, raised Hindu, uses story of Chri..."
4,SATAN 2: Russia unvelis an image of its terrif...,"The RS-28 Sarmat missile, dubbed Satan 2, will...",1,SATAN 2: Russia unvelis an image of its terrif...


In [14]:
(data["news"] == '').all()

np.False_

In [15]:
data.shape

(72134, 4)

In [16]:
data['news'].duplicated().sum()



np.int64(8456)

In [17]:
data.duplicated().sum()

np.int64(8456)

In [18]:
data.drop_duplicates(inplace = True)

In [19]:
data.shape

(63678, 4)

In [20]:
data['label'].value_counts()

,count
label,
0,34791
1,28887


In [21]:
def text_preprocessing(text):
  text = text.lower()

  text = re.sub(r'https?://\S+|www\.\S+','',text)

  text = re.sub(r'<.*?>','',text)

  text = re.sub(r'[^\w\s]','',text)

  text = re.sub(r'\d','',text)

  text = re.sub(r'\n','',text)

  return text

In [22]:
data['news'] = data['news'].apply(text_preprocessing)

In [23]:
X = data['news']
y = data['label']

In [24]:
from sklearn.model_selection import train_test_split

X_train,X_test,y_train,y_test = train_test_split(X,y,test_size = 0.2, random_state = 42)

In [25]:
X_train.shape

(50942,)

In [26]:
X_test.shape

(12736,)

In [27]:
from sklearn.feature_extraction.text import CountVectorizer

cv = CountVectorizer(
    max_features=5000
)

X_train_cv = cv.fit_transform(X_train)
X_test_cv = cv.transform(X_test)

In [28]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators = 100,
    max_depth = 5,
    random_state = 42
)

In [29]:
rf.fit(X_train_cv,y_train)

RandomForestClassifier(max_depth=5, random_state=42)

In [30]:
y_pred = rf.predict(X_test_cv)

In [31]:
from sklearn.metrics import accuracy_score
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report

print(f"accuracy: {accuracy_score(y_test,y_pred)}")
print(confusion_matrix(y_test,y_pred))
print(classification_report(y_test,y_pred))

accuracy: 0.8877983668341709
[[6798  227]
 [1202 4509]]
              precision    recall  f1-score   support

           0       0.85      0.97      0.90      7025
           1       0.95      0.79      0.86      5711

    accuracy                           0.89     12736
   macro avg       0.90      0.88      0.88     12736
weighted avg       0.90      0.89      0.89     12736



using random forest and count vectorizer got 88.78%

In [32]:
def prediction(text):

  text = text_preprocessing(text)
  text_vector = cv.transform([text])
  return rf.predict(text_vector)

In [33]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(
    max_features = 5000,
    ngram_range = (1,2)
)

X_train_tf = tfidf.fit_transform(X_train)
X_test_tf = tfidf.transform(X_test)

In [34]:
rf2 = RandomForestClassifier(
    n_estimators = 100,
    max_depth = 5,
    random_state = 42
)

rf2.fit(X_train_tf,y_train)

RandomForestClassifier(max_depth=5, random_state=42)

In [35]:
y_pred = rf2.predict(X_test_tf)

In [36]:
accuracy_score(y_test,y_pred)

0.9064070351758794

using RandomForestClassifier and tfidf got 90.64%

In [37]:
confusion_matrix(y_test,y_pred)

array([[6581,  444],
       [ 748, 4963]])

In [38]:
print(classification_report(y_test,y_pred))

              precision    recall  f1-score   support

           0       0.90      0.94      0.92      7025
           1       0.92      0.87      0.89      5711

    accuracy                           0.91     12736
   macro avg       0.91      0.90      0.90     12736
weighted avg       0.91      0.91      0.91     12736



In [39]:
from sklearn.linear_model import LogisticRegression

lr = LogisticRegression(max_iter = 1000)

lr.fit(X_train_tf,y_train)

LogisticRegression(max_iter=1000)

In [40]:
y_pred = lr.predict(X_test_tf)

accuracy_score(y_test,y_pred)

0.9568153266331658

using LogisticRegression and tfidf got 95.68%

In [41]:
confusion_matrix(y_test,y_pred)

array([[6751,  274],
       [ 276, 5435]])

In [43]:
from sklearn.model_selection import GridSearchCV

rf = RandomForestClassifier(random_state=42)

param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [10, 20],
    'min_samples_split': [2, 5]
}

grid = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid,
    cv=3,
    scoring='f1',
    n_jobs=-1
)



grid.fit(X_train_tf, y_train)

GridSearchCV(cv=3, estimator=RandomForestClassifier(random_state=42), n_jobs=-1,
             param_grid={'max_depth': [10, 20], 'min_samples_split': [2, 5],
                         'n_estimators': [100, 200]},
             scoring='f1')

In [44]:
print(grid.best_params_)
print(grid.best_score_)

{'max_depth': 20, 'min_samples_split': 5, 'n_estimators': 200}
0.9347150612158238


using GridSearchCV, RandomForestClassifier and tfidf got 93.47%

In [45]:
vocab = {
    '<PAD>' : 0,
    '<UNK>':1
    }

def add_vocab(text):
  for word in text.split():
    if word not in vocab:
      vocab[word] = len(vocab)

In [46]:
data['news'].apply(add_vocab)

,news
0,None
1,None
2,None
3,None
4,None
...,...
72127,None
72129,None
72130,None
72131,None


In [47]:
len(vocab)

401795

In [48]:
# convert word to indices

def text_to_indices(text,vocab):
  texts = []
  text = text_preprocessing(text)
  for word in text.split():
    if word in vocab:
      texts.append(vocab[word])
    else:
      texts.append(vocab['<UNK>'])
  return texts

In [49]:
text_to_indices('Narendra modi is the prime minister of india',vocab)

[17447, 17448, 20, 27, 1602, 658, 26, 814]

In [50]:
from torch.utils.data import Dataset, DataLoader

In [51]:
max_len = 300

def pad_sequences(text,vocab):
  if(len(text) < max_len):
    text = text + [vocab['<PAD>']] * (max_len - len(text))

  else:
    text = text[:max_len]

  return text

In [52]:
class MyDataset(Dataset):

  def __init__(self,data,vocab):
    self.data = data
    self.vocab = vocab

  def __len__(self):
    return self.data.shape[0]

  def __getitem__(self,index):
    numerical_data = text_to_indices(self.data.iloc[index]['news'], self.vocab)
    padded_data = pad_sequences(numerical_data, self.vocab)
    label = self.data.iloc[index]['label']

    return torch.tensor(padded_data, dtype = torch.long), torch.tensor(label, dtype = torch.float32)

In [53]:
train_df, test_df = train_test_split(
    data,
    test_size=0.2,
    random_state=42,
    stratify=data["label"]
)

In [54]:
train_dataset = MyDataset(train_df, vocab)
test_dataset = MyDataset(test_df, vocab)

In [55]:
train_dataset[0][0].shape

torch.Size([300])

In [56]:
train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False
)

In [57]:
vocab_2 = {
    '<PAD>' : 0,
    '<UNK>' : 1
}

def build_vocab(text):
  for word in text.split():
    if word not in vocab_2:
      vocab_2[word] = len(vocab_2)

train_df['news'].apply(build_vocab)

,news
27152,None
4980,None
8537,None
9369,None
29322,None
...,...
48585,None
12966,None
35202,None
47427,None


In [58]:
len(vocab_2)

348091

In [59]:
train_dataset_2 = MyDataset(train_df,vocab_2)
test_dataset_2 = MyDataset(test_df,vocab_2)

In [60]:
train_loader_2 = DataLoader(
    train_dataset_2,
    batch_size=32,
    shuffle=True
)

test_loader_2 = DataLoader(
    test_dataset_2,
    batch_size=32,
    shuffle=False
)

In [61]:
class LSTMClassifier(nn.Module):

  def __init__(self,vocab_size,embedding_dim,hidden_dim):
    super().__init__()
    self.embedding = nn.Embedding(vocab_size, embedding_dim)
    self.lstm = nn.LSTM(embedding_dim, hidden_dim, batch_first = True)
    self.fc = nn.Linear(hidden_dim,1)
    self.sigmoid = nn.Sigmoid()

  def forward(self, x):
    embedded = self.embedding(x)
    intermediate_hidden_state, (final_hidden_state, final_cell_state) = self.lstm(embedded)
    final_hidden_state = final_hidden_state[-1]
    output = self.fc(final_hidden_state)
    output = self.sigmoid(output)
    return output

In [62]:
model = LSTMClassifier(len(vocab_2),100,128)

In [63]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [64]:
model.to(device)

LSTMClassifier(
  (embedding): Embedding(348091, 100)
  (lstm): LSTM(100, 128, batch_first=True)
  (fc): Linear(in_features=128, out_features=1, bias=True)
  (sigmoid): Sigmoid()
)

In [65]:
epochs = 10
learning_rate = 0.001

loss_function = nn.BCELoss()

optimizer = torch.optim.Adam(model.parameters(), lr = learning_rate)

In [70]:
for epoch in range(epochs):
  model.train()

  total_loss = 0

  for batch_X, batch_y in train_loader_2:

    batch_X, batch_y = batch_X.to(device), batch_y.to(device)

    optimizer.zero_grad()

    output = model(batch_X)

    loss = loss_function(output.squeeze(), batch_y)

    loss.backward()

    optimizer.step()

    total_loss += loss.item()

  print(f'epochs: {epoch + 1}, loss: {total_loss:.4f}')


epochs: 1, loss: 19.8174
epochs: 2, loss: 14.7021
epochs: 3, loss: 9.2171
epochs: 4, loss: 5.8113
epochs: 5, loss: 6.1072
epochs: 6, loss: 3.3252
epochs: 7, loss: 3.6520
epochs: 8, loss: 2.6333
epochs: 9, loss: 1.7744
epochs: 10, loss: 0.2407


In [71]:
model.eval()

correct = 0
total = 0

with torch.no_grad():

    for batch_X, batch_y in test_loader_2:

        batch_X = batch_X.to(device)
        batch_y = batch_y.to(device)

        output = model(batch_X)

        preds = (output.squeeze() >= 0.5).float()

        total += batch_y.size(0)
        correct += (preds == batch_y).sum().item()

accuracy = correct / total

print(f"Accuracy: {accuracy:.4f}")

Accuracy: 0.9698


LSTM Accuracy = 96.74% and 96.98%

In [72]:
class LSTMClassifier2(nn.Module):

  def __init__(self,vocab_size,embedding_dim,hidden_dim):
    super().__init__()
    self.embedding = nn.Embedding(vocab_size, embedding_dim)
    self.lstm = nn.LSTM(embedding_dim, hidden_dim, dropout=0.3, batch_first = True)
    self.fc = nn.Linear(hidden_dim,1)
    self.sigmoid = nn.Sigmoid()

  def forward(self, x):
    embedded = self.embedding(x)
    intermediate_hidden_state, (final_hidden_state, final_cell_state) = self.lstm(embedded)
    final_hidden_state = final_hidden_state[-1]
    output = self.fc(final_hidden_state)
    output = self.sigmoid(output)
    return output

In [73]:
model2 = LSTMClassifier2(len(vocab_2),100,128)

/usr/local/lib/python3.12/dist-packages/torch/nn/modules/rnn.py:1013: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.3 and num_layers=1
  super().__init__("LSTM", *args, **kwargs)


In [74]:
model2.to(device)

LSTMClassifier2(
  (embedding): Embedding(348091, 100)
  (lstm): LSTM(100, 128, batch_first=True, dropout=0.3)
  (fc): Linear(in_features=128, out_features=1, bias=True)
  (sigmoid): Sigmoid()
)

In [75]:
epochs = 10
learning_rate = 0.001

loss_function = nn.BCELoss()

optimizer2 = torch.optim.Adam(model2.parameters(), lr = learning_rate)

In [76]:
for epoch in range(epochs):
  model2.train()

  total_loss = 0

  for batch_X, batch_y in train_loader_2:

    batch_X, batch_y = batch_X.to(device), batch_y.to(device)

    optimizer2.zero_grad()

    output = model2(batch_X)

    loss = loss_function(output.squeeze(), batch_y)

    loss.backward()

    optimizer2.step()

    total_loss += loss.item()

  print(f'epochs: {epoch + 1}, loss: {total_loss:.4f}')

epochs: 1, loss: 951.4176
epochs: 2, loss: 332.3758
epochs: 3, loss: 159.0651
epochs: 4, loss: 88.0737
epochs: 5, loss: 57.9898
epochs: 6, loss: 37.7046
epochs: 7, loss: 24.5891
epochs: 8, loss: 17.4218
epochs: 9, loss: 15.5930
epochs: 10, loss: 13.7182


In [77]:
model2.eval()

correct2 = 0
total2 = 0

with torch.no_grad():

    for batch_X, batch_y in test_loader_2:

        batch_X = batch_X.to(device)
        batch_y = batch_y.to(device)

        output = model2(batch_X)

        preds = (output.squeeze() >= 0.5).float()

        total2 += batch_y.size(0)
        correct2 += (preds == batch_y).sum().item()

accuracy2 = correct2 / total2

print(f"Accuracy: {accuracy2:.4f}")

Accuracy: 0.9604


LSTM with 0.3 Dropouts got 96.04%

In [88]:
class LSTMClassifier3(nn.Module):

  def __init__(self,vocab_size,embedding_dim,hidden_dim):
    super().__init__()
    self.embedding = nn.Embedding(vocab_size, embedding_dim)
    self.lstm = nn.LSTM(embedding_dim, hidden_dim, dropout=0.3, bidirectional = True, batch_first = True)
    self.fc = nn.Linear(hidden_dim * 2,1)
    self.sigmoid = nn.Sigmoid()

  def forward(self, x):
    embedded = self.embedding(x)
    intermediate_hidden_state, (final_hidden_state, final_cell_state) = self.lstm(embedded)
    forward_hidden_state = final_hidden_state[0]
    backward_hidden_state = final_hidden_state[1]
    hidden = torch.cat((forward_hidden_state,backward_hidden_state), dim = 1)
    output = self.fc(hidden)
    output = self.sigmoid(output)
    return output

In [89]:
model3 = LSTMClassifier3(len(vocab_2),100,128)

/usr/local/lib/python3.12/dist-packages/torch/nn/modules/rnn.py:1013: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.3 and num_layers=1
  super().__init__("LSTM", *args, **kwargs)


In [90]:
model3.to(device)

LSTMClassifier3(
  (embedding): Embedding(348091, 100)
  (lstm): LSTM(100, 128, batch_first=True, dropout=0.3, bidirectional=True)
  (fc): Linear(in_features=256, out_features=1, bias=True)
  (sigmoid): Sigmoid()
)

In [91]:
epochs = 10
learning_rate = 0.001

loss_function = nn.BCELoss()

optimizer3 = torch.optim.Adam(model3.parameters(), lr = learning_rate)

In [92]:
for epoch in range(epochs):
  model3.train()

  total_loss = 0

  for batch_X, batch_y in train_loader_2:

    batch_X, batch_y = batch_X.to(device), batch_y.to(device)

    optimizer3.zero_grad()

    output = model3(batch_X)

    loss = loss_function(output.squeeze(), batch_y)

    loss.backward()

    optimizer3.step()

    total_loss += loss.item()

  print(f'epochs: {epoch + 1}, loss: {total_loss:.4f}')

epochs: 1, loss: 416.3325
epochs: 2, loss: 178.5291
epochs: 3, loss: 78.4134
epochs: 4, loss: 34.3450
epochs: 5, loss: 11.9878
epochs: 6, loss: 6.4979
epochs: 7, loss: 4.3687
epochs: 8, loss: 2.5187
epochs: 9, loss: 1.8485
epochs: 10, loss: 3.0394


In [93]:
model3.eval()

correct3 = 0
total3 = 0

with torch.no_grad():

    for batch_X, batch_y in test_loader_2:

        batch_X = batch_X.to(device)
        batch_y = batch_y.to(device)

        output = model3(batch_X)

        preds = (output.squeeze() >= 0.5).float()

        total3 += batch_y.size(0)
        correct3 += (preds == batch_y).sum().item()

accuracy3 = correct3 / total3

print(f"Accuracy: {accuracy3:.4f}")

Accuracy: 0.9689


Bi-LSTM with 0.3 Dropouts - 96.89%